# VAE

In [4]:
from dataclasses import dataclass
from typing import Dict, Tuple
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------
# Config
# ----------------------------

@dataclass
class VAEConfig:
    in_dim: int = 16          # for toy data; change to 784 for MNIST flattened
    hidden: int = 64
    z_dim: int = 8
    lr: float = 1e-3
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# ----------------------------
# Encoder / Decoder
# ----------------------------

class MLPEncoder(nn.Module):
    """
    x: (B, in_dim) -> (mu, logvar): (B, z_dim) each
    """
    def __init__(self, in_dim: int, hidden: int, z_dim: int):
        super().__init__()
        self.net = nn.Sequential( # two layers
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
        )
        self.mu = nn.Linear(hidden, z_dim)
        self.logvar = nn.Linear(hidden, z_dim)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.net(x)
        return self.mu(h), self.logvar(h) # the learned mean and variance


class MLPDecoder(nn.Module):
    """
    z: (B, z_dim) -> x_hat: (B, in_dim)
    For bounded [0,1] data, use Sigmoid; for real-valued, no activation.
    """
    def __init__(self, z_dim: int, hidden: int, out_dim: int, bounded: bool = False):
        super().__init__()
        layers = [
            nn.Linear(z_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, out_dim),
        ]
        if bounded:
            layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)

In [12]:
# ----------------------------
# VAE (candidate implements core bits)
# ----------------------------

class VAE(nn.Module):
    def __init__(self, cfg: VAEConfig, bounded_data: bool = False):
        super().__init__()
        self.cfg = cfg
        self.encoder = MLPEncoder(cfg.in_dim, cfg.hidden, cfg.z_dim)
        self.decoder = MLPDecoder(cfg.z_dim, cfg.hidden, cfg.in_dim, bounded=bounded_data)

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor, eps: torch.Tensor = None) -> torch.Tensor:
        """
        TODO: implement reparameterization:
            z = mu + sigma * eps

        We allow passing eps to make tests deterministic.
        """
        if eps is None:
            eps = torch.randn_like(mu)
        # ---- TODO (candidate): replace below with correct formula ----
        # pass

        return mu + torch.exp(0.5 * logvar) * eps

    @staticmethod
    def kl_standard_normal(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """
        KL(q(z|x) || N(0,I)) for diagonal Gaussian.
        Returns scalar batch-mean KL.
        """
        # ---- TODO (candidate): implement correctly ----
        # pass
        return 0.5 * torch.sum(mu ** 2 + torch.exp(logvar) - 1 - logvar, dim=-1).mean()

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        # ---- TODO (candidate): implement correctly ----
        # pass

        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)

        return {"x_hat": x_hat, "mu": mu, "logvar": logvar, "z": z}


def reconstruction_loss_mse(x_hat: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    return F.mse_loss(x_hat, x)

def elbo_loss(x_hat: torch.Tensor, x: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor, beta: float = 1.0) -> Dict[str, torch.Tensor]:
    # ---- TODO (candidate): implement correctly ----
    # pass
    rec = reconstruction_loss_mse(x_hat, x)
    kl = VAE.kl_standard_normal(mu, logvar)
    total = rec + beta * kl

    return {"total": total, "recon": rec, "kl": kl}


In [13]:
# ============================================================
# Tests (pytest-style but runnable via python)
# ============================================================

def test_shapes():
    torch.manual_seed(0)
    cfg = VAEConfig(in_dim=16, hidden=32, z_dim=8, device="cpu")
    model = VAE(cfg).to(cfg.device)

    x = torch.randn(4, cfg.in_dim)
    out = model(x)
    assert out["x_hat"].shape == x.shape
    assert out["mu"].shape == (4, cfg.z_dim)
    assert out["logvar"].shape == (4, cfg.z_dim)
    assert out["z"].shape == (4, cfg.z_dim)

def test_reparameterize_deterministic():
    """
    If eps is provided, z should exactly match mu + exp(0.5*logvar)*eps.
    """
    torch.manual_seed(0)
    mu = torch.tensor([[1.0, -1.0]])
    logvar = torch.tensor([[math.log(4.0), math.log(0.25)]])  # sigma^2 = [4, .25] => sigma=[2, .5]
    eps = torch.tensor([[0.5, -2.0]])
    z = VAE.reparameterize(mu, logvar, eps=eps)
    expected = torch.tensor([[1.0 + 2.0 * 0.5, -1.0 + 0.5 * (-2.0)]])
    assert torch.allclose(z, expected, atol=0, rtol=0)

def test_kl_zero_when_standard_normal():
    """
    If mu=0 and logvar=0 => q(z|x)=N(0,I) => KL=0
    """
    mu = torch.zeros(10, 7)
    logvar = torch.zeros(10, 7)
    kl = VAE.kl_standard_normal(mu, logvar)
    assert torch.allclose(kl, torch.tensor(0.0), atol=1e-7)

def test_kl_matches_manual_single_sample():
    mu = torch.tensor([[1.0, 2.0]])
    logvar = torch.tensor([[0.0, math.log(4.0)]])  # var=[1,4]
    # manual: 0.5 * sum(exp(logvar) + mu^2 -1 -logvar)
    manual = 0.5 * ((1.0 + 1.0**2 - 1.0 - 0.0) + (4.0 + 2.0**2 - 1.0 - math.log(4.0)))
    kl = VAE.kl_standard_normal(mu, logvar)
    assert torch.allclose(kl, torch.tensor(manual), atol=1e-6)

def test_grad_flow_through_mu_logvar():
    """
    Ensure loss backprop produces grads for encoder outputs (mu/logvar),
    meaning reparameterization + KL are wired correctly.
    """
    torch.manual_seed(0)
    cfg = VAEConfig(in_dim=16, hidden=32, z_dim=8, device="cpu")
    model = VAE(cfg).to(cfg.device)

    x = torch.randn(4, cfg.in_dim)
    out = model(x)
    losses = elbo_loss(out["x_hat"], x, out["mu"], out["logvar"])

    # retain grads on intermediate tensors
    out["mu"].retain_grad()
    out["logvar"].retain_grad()

    losses["total"].backward()

    assert out["mu"].grad is not None and torch.isfinite(out["mu"].grad).all()
    assert out["logvar"].grad is not None and torch.isfinite(out["logvar"].grad).all()

def test_loss_decreases_on_toy_data():
    """
    Not a strict guarantee, but usually passes with correct wiring.
    We train briefly on a fixed toy dataset.
    """
    torch.manual_seed(0)
    cfg = VAEConfig(in_dim=16, hidden=64, z_dim=4, lr=5e-3, device="cpu")
    model = VAE(cfg).to(cfg.device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    # Toy data: mixture of two Gaussians in x-space
    n = 256
    x0 = torch.randn(n//2, cfg.in_dim) * 0.5 + 1.0
    x1 = torch.randn(n//2, cfg.in_dim) * 0.5 - 1.0
    X = torch.cat([x0, x1], dim=0)

    def step():
        out = model(X)
        losses = elbo_loss(out["x_hat"], X, out["mu"], out["logvar"], beta=0.1)
        opt.zero_grad(set_to_none=True)
        losses["total"].backward()
        opt.step()
        return losses["total"].item()

    l0 = step()
    for _ in range(50):
        l1 = step()

    # Expect some improvement
    assert l1 < l0, (l0, l1)

def run_all_tests():
    for fn in [
        test_shapes,
        test_reparameterize_deterministic,
        test_kl_zero_when_standard_normal,
        test_kl_matches_manual_single_sample,
        test_grad_flow_through_mu_logvar,
        test_loss_decreases_on_toy_data,
    ]:
        fn()
    print("All VAE tests passed.")


if __name__ == "__main__":
    run_all_tests()

All VAE tests passed.


## Solutions

```python
"""
VAE interview scaffold + tests (PyTorch)

What this gives you:
- A minimal VAE model skeleton with deliberate TODOs
- Deterministic unit tests that validate:
    (1) shapes
    (2) reparameterization correctness
    (3) KL formula correctness
    (4) gradient flow through mu/logvar
    (5) ELBO decreases a bit on a toy dataset

How to use in an interview:
- Hand them vae_scaffold.py and test_vae.py (below) OR keep as one file.
- Ask them to implement TODOs until tests pass.
"""

from dataclasses import dataclass
from typing import Dict, Tuple
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------
# Config
# ----------------------------

@dataclass
class VAEConfig:
    in_dim: int = 16          # for toy data; change to 784 for MNIST flattened
    hidden: int = 64
    z_dim: int = 8
    lr: float = 1e-3
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# ----------------------------
# Encoder / Decoder
# ----------------------------

class MLPEncoder(nn.Module):
    """
    x: (B, in_dim) -> (mu, logvar): (B, z_dim) each
    """
    def __init__(self, in_dim: int, hidden: int, z_dim: int):
        super().__init__()
        self.net = nn.Sequential( # two layers
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
        )
        self.mu = nn.Linear(hidden, z_dim)
        self.logvar = nn.Linear(hidden, z_dim)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.net(x)
        return self.mu(h), self.logvar(h) # the learned mean and variance


class MLPDecoder(nn.Module):
    """
    z: (B, z_dim) -> x_hat: (B, in_dim)
    For bounded [0,1] data, use Sigmoid; for real-valued, no activation.
    """
    def __init__(self, z_dim: int, hidden: int, out_dim: int, bounded: bool = False):
        super().__init__()
        layers = [
            nn.Linear(z_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, out_dim),
        ]
        if bounded:
            layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


# ----------------------------
# VAE (candidate implements core bits)
# ----------------------------

class VAE(nn.Module):
    def __init__(self, cfg: VAEConfig, bounded_data: bool = False):
        super().__init__()
        self.cfg = cfg
        self.encoder = MLPEncoder(cfg.in_dim, cfg.hidden, cfg.z_dim)
        self.decoder = MLPDecoder(cfg.z_dim, cfg.hidden, cfg.in_dim, bounded=bounded_data)

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor, eps: torch.Tensor = None) -> torch.Tensor:
        """
        TODO: implement reparameterization:
            z = mu + sigma * eps

        We allow passing eps to make tests deterministic.
        """
        if eps is None:
            eps = torch.randn_like(mu)
        # ---- TODO (candidate): replace below with correct formula ----
        sigma = torch.exp(0.5 * logvar)
        z = mu + sigma * eps
        return z

    @staticmethod
    def kl_standard_normal(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """
        KL(q(z|x) || N(0,I)) for diagonal Gaussian.
        Returns scalar batch-mean KL.
        """
        # ---- TODO (candidate): implement correctly ----
        kl_per_sample = 0.5 * torch.sum(torch.exp(logvar) + mu**2 - 1.0 - logvar, dim=-1)
        return kl_per_sample.mean()

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        # ---- TODO (candidate): implement correctly ----
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return {"x_hat": x_hat, "mu": mu, "logvar": logvar, "z": z}


def reconstruction_loss_mse(x_hat: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    return F.mse_loss(x_hat, x)

def elbo_loss(x_hat: torch.Tensor, x: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor, beta: float = 1.0) -> Dict[str, torch.Tensor]:
    # ---- TODO (candidate): implement correctly ----
    rec = reconstruction_loss_mse(x_hat, x)
    kl = VAE.kl_standard_normal(mu, logvar)
    total = rec + beta * kl
    return {"total": total, "recon": rec, "kl": kl}
```